# PyTorch Deep Learning Baseline: Multilayer Perceptron (MLP) with TensorBoard

**Task:** Predict building electrical power consumption 24 hours ahead (144 steps at 10-minute resolution) using a deep PyTorch Multilayer Perceptron (MLP).  
**Data:** `library_featured_v2.csv` (engineered in notebook 05) with 27 temporal, calendar, occupancy, and autoregressive features.  
**Benchmark Target:** Compare against tree-based baselines (LightGBM, Extra Trees, XGBoost) using the exact same chronological 70/30 test split.

---

### Architectural Formulation & Regression Output
In deep learning for building energy forecasting:
1. **Network Depth & Regularization:** We employ a 3-layer deep architecture with batch normalization (`BatchNorm1d`) and dropout (`p=0.3`) between layers to prevent overfitting to seasonal weather fluctuations.
2. **Output Layer (Linear vs Softmax):**
   - **Why NOT Softmax?** Softmax normalizes an output vector into a probability distribution where values are bounded between 0 and 1 and sum to 1. It is designed for multi-class classification.
   - **Why Linear (No Activation)?** Electrical power is a continuous, unbounded physical quantity (measured in Watts: $0$ to $\sim 20{,}000\text{ W}$). The final layer must be an unconstrained linear projection: $\hat{y} = \mathbf{w}^T \mathbf{h} + b$.
3. **Comprehensive TensorBoard Monitoring:** Every epoch records training/validation loss curves, learning rate dynamics, gradient histograms, actual vs. predicted time-series overlays, and final hyperparameters in the unified `runs/` directory for committee review.

## 1. Imports & TensorBoard Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure utils path is discoverable
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
if "." not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from utils.tb_logger import TorchTBLogger

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Computation Device: {DEVICE}")

PyTorch Version: 2.14.0+cpu
Computation Device: cpu


## 2. Load Engineered Feature Matrix

We load `library_featured_v2.csv` containing the 27 continuous, cyclical, calendar, and autoregressive lag features created in notebook 05.

In [2]:
DATA_PATH = "library_featured_v2.csv" if os.path.exists("library_featured_v2.csv") else "../library_featured_v2.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded library_featured_v2.csv: 41,248 rows x 37 columns


## 3. Feature Selection & Chronological Train/Val/Test Split

To maintain exact scientific parity with the tree models:
- **Test Set:** The final 30% (12,375 rows) is held out strictly for final evaluation.
- **Train & Validation Sets:** The first 70% (28,873 rows) is split chronologically into:
  - **Train Sub-split (85%):** 24,542 rows used to optimize neural network weights.
  - **Validation Sub-split (15%):** 4,331 rows used for learning rate scheduling, early stopping, and checkpoint selection without peeking at the test set.

In [3]:
FEATURE_COLUMNS = [
    # Baseline power & occupancy
    "power", "occupancy_count", "occupancy_roll_mean_6", "occupancy_diff_1", "rooms_occupied_est",
    # Cyclical temporal features
    "hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    # BMS operational schedule flags
    "weekend", "is_class_hour", "is_evening_study", "is_night", "is_exam_month", "is_vacation_month",
    # Autoregressive lags & rolling stats
    "power_lag_144", "power_lag_288", "power_lag_1008", "occupancy_lag_144",
    "power_roll_mean_144", "power_roll_std_144", "power_roll_min_144", "power_roll_max_144"
]
TARGET_COLUMN = "target"

# Primary 70 / 30 chronological split
test_split_idx = int(len(df) * 0.70)
train_val_df = df.iloc[:test_split_idx].reset_index(drop=True)
test_df = df.iloc[test_split_idx:].reset_index(drop=True)

# Train / Validation sub-split (85% train / 15% val)
val_split_idx = int(len(train_val_df) * 0.85)
train_df = train_val_df.iloc[:val_split_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_split_idx:].reset_index(drop=True)

print(f"Features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print(f"Train sub-split: {len(train_df):>6,} rows ({train_df['timestamp'].min()} to {train_df['timestamp'].max()})")
print(f"Validation split:{len(val_df):>6,} rows ({val_df['timestamp'].min()} to {val_df['timestamp'].max()})")
print(f"Test split:      {len(test_df):>6,} rows ({test_df['timestamp'].min()} to {test_df['timestamp'].max()})")

Features (27): ['power', 'occupancy_count', 'occupancy_roll_mean_6', 'occupancy_diff_1', 'rooms_occupied_est', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos', 'weekend', 'is_class_hour', 'is_evening_study', 'is_night', 'is_exam_month', 'is_vacation_month', 'power_lag_144', 'power_lag_288', 'power_lag_1008', 'occupancy_lag_144', 'power_roll_mean_144', 'power_roll_std_144', 'power_roll_min_144', 'power_roll_max_144']
Train sub-split: 24,542 rows (2014-02-26 10:30:00+05:30 to 2016-08-10 12:00:00+05:30)
Validation split: 4,331 rows (2016-08-10 12:10:00+05:30 to 2016-11-20 12:50:00+05:30)
Test split:      12,375 rows (2016-11-20 13:00:00+05:30 to 2017-11-02 23:50:00+05:30)


## 4. Feature and Target Normalization

Neural networks require zero-mean, unit-variance scaled inputs for stable gradient backpropagation.
- `scaler_x` is fitted **strictly on the training sub-split** and applied to validation and test sets.
- `scaler_y` is fitted on the training target, allowing us to invert test predictions back to true physical Watts ($W$) for metric evaluation.

In [4]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_raw = train_df[FEATURE_COLUMNS].values
y_train_raw = train_df[TARGET_COLUMN].values.reshape(-1, 1)

X_val_raw = val_df[FEATURE_COLUMNS].values
y_val_raw = val_df[TARGET_COLUMN].values.reshape(-1, 1)

X_test_raw = test_df[FEATURE_COLUMNS].values
y_test_raw = test_df[TARGET_COLUMN].values.reshape(-1, 1)

# Fit exclusively on train set
X_train = scaler_x.fit_transform(X_train_raw)
y_train = scaler_y.fit_transform(y_train_raw).ravel()

X_val = scaler_x.transform(X_val_raw)
y_val = scaler_y.transform(y_val_raw).ravel()

X_test = scaler_x.transform(X_test_raw)
y_test = scaler_y.transform(y_test_raw).ravel()

# PyTorch Datasets and DataLoaders
BATCH_SIZE = 128

train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
val_dataset   = TensorDataset(torch.FloatTensor(X_val),   torch.FloatTensor(y_val))
test_dataset  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders prepared: Train batches={len(train_loader)}, Val batches={len(val_loader)}, Test batches={len(test_loader)}")

DataLoaders prepared: Train batches=192, Val batches=34, Test batches=97


## 5. PyTorch MLP Model Architecture

The architecture consists of:
- Input projection: $\mathbb{R}^{27} \to \mathbb{R}^{256}$
- Hidden layer 1: `Linear(256)` + `BatchNorm1d` + `ReLU` + `Dropout(0.3)`
- Hidden layer 2: `Linear(128)` + `BatchNorm1d` + `ReLU` + `Dropout(0.3)`
- Hidden layer 3: `Linear(64)` + `BatchNorm1d` + `ReLU` + `Dropout(0.2)`
- Output layer: `Linear(1)` — Linear projection with **no activation** for continuous regression.

In [5]:
class BMS_MLP(nn.Module):
    def __init__(self, in_features, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = in_features
        for i, h_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            drop_rate = dropout if i < len(hidden_dims) - 1 else 0.2
            layers.append(nn.Dropout(drop_rate))
            prev_dim = h_dim
        
        # Final output layer: Linear (NO ACTIVATION) for continuous regression
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x).squeeze(-1)

model = BMS_MLP(in_features=len(FEATURE_COLUMNS), hidden_dims=[256, 128, 64], dropout=0.3).to(DEVICE)
print(model)
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

BMS_MLP(
  (network): Sequential(
    (0): Linear(in_features=27, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=64, out_features=1, bias=True)
  )
)
Total Trainable Parameters: 49,281


## 6. Initialize TensorBoard Logger & Computational Graph

We initialize `TorchTBLogger`, logging to `runs/MLP`. Passing a sample batch logs the neural network computational graph directly to the TensorBoard **Graphs** tab.

In [6]:
log_root = "../runs" if os.path.basename(os.getcwd()) == "notebooks" else "runs"
os.makedirs(log_root, exist_ok=True)

# Sample batch for graph logging
sample_x, _ = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)

tb = TorchTBLogger(
    model_name="MLP",
    model=model,
    sample_input=sample_x,
    log_root=log_root
)

[TB] 'MLP' -> ../runs\MLP
[TB] Launch: tensorboard --logdir=../runs/


[TB] Model graph logged.


## 7. Model Training & Validation with TensorBoard Per-Epoch Logging

We train for 30 epochs using:
- **Optimizer:** `AdamW` with initial learning rate $\eta = 10^{-3}$ and weight decay $\lambda = 10^{-4}$.
- **Loss Criterion:** Mean Squared Error (`nn.MSELoss`).
- **Learning Rate Scheduler:** `CosineAnnealingLR` decaying over the 30 epochs.
- **Gradient Clipping:** Max norm of $1.0$ to ensure stability.
- **Model Checkpointing:** Saves the model state dictionary when validation loss reaches a new minimum.

In [7]:
EPOCHS = 30
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

best_val_loss = float("inf")
checkpoint_path = os.path.join(log_root, "best_MLP.pt")

print(f"Beginning MLP training on {DEVICE} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    # ── Training Phase ──
    model.train()
    train_losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        
    avg_train_loss = np.mean(train_losses)
    
    # ── Validation Phase ──
    model.eval()
    val_losses = []
    val_preds_list = []
    val_targets_list = []
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            val_losses.append(loss.item())
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(y_b.cpu().numpy())
            
    avg_val_loss = np.mean(val_losses)
    current_lr = scheduler.get_last_lr()[0]
    
    # Convert predictions back to Watts to record physical metrics in TensorBoard
    val_preds_W = scaler_y.inverse_transform(np.array(val_preds_list).reshape(-1, 1)).ravel()
    val_targets_W = scaler_y.inverse_transform(np.array(val_targets_list).reshape(-1, 1)).ravel()
    
    val_mae = mean_absolute_error(val_targets_W, val_preds_W)
    val_rmse = np.sqrt(mean_squared_error(val_targets_W, val_preds_W))
    val_r2 = r2_score(val_targets_W, val_preds_W)
    
    # Log to TensorBoard Scalars
    tb.log_epoch(
        epoch=epoch,
        train_loss=avg_train_loss,
        val_loss=avg_val_loss,
        y_val=val_targets_W,
        y_pred=val_preds_W,
        lr=current_lr
    )
    
    # Log gradient histograms every 10 epochs
    if epoch % 10 == 0:
        tb.log_gradients(model, epoch)
        
    # Checkpoint best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        
    scheduler.step()
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>2d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val MAE: {val_mae:,.1f} W | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")

print(f"\nTraining complete. Best checkpoint saved to {checkpoint_path}")

Beginning MLP training on cpu for 30 epochs...


Epoch  1/30 | Train Loss: 0.3994 | Val Loss: 0.3880 | Val MAE: 3,016.3 W | Val R²: 0.6715 | LR: 1.00e-03


Epoch  5/30 | Train Loss: 0.2099 | Val Loss: 0.3608 | Val MAE: 2,834.4 W | Val R²: 0.6944 | LR: 9.57e-04


Epoch 10/30 | Train Loss: 0.1691 | Val Loss: 0.3743 | Val MAE: 2,914.0 W | Val R²: 0.6831 | LR: 7.96e-04


Epoch 15/30 | Train Loss: 0.1472 | Val Loss: 0.4104 | Val MAE: 3,096.4 W | Val R²: 0.6524 | LR: 5.57e-04


Epoch 20/30 | Train Loss: 0.1386 | Val Loss: 0.4022 | Val MAE: 3,060.5 W | Val R²: 0.6596 | LR: 3.04e-04


Epoch 25/30 | Train Loss: 0.1328 | Val Loss: 0.3999 | Val MAE: 3,064.1 W | Val R²: 0.6615 | LR: 1.05e-04


Epoch 30/30 | Train Loss: 0.1304 | Val Loss: 0.4012 | Val MAE: 3,061.0 W | Val R²: 0.6604 | LR: 1.27e-05

Training complete. Best checkpoint saved to ../runs\best_MLP.pt


## 8. Final Test Evaluation & Benchmark Comparison

We load the optimal model checkpoint, evaluate on the unseen 12,375 test rows, and invert back to physical Watts ($W$).

In [8]:
# Load optimal weights
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

# Predict on test set
test_preds_list = []
with torch.no_grad():
    for x_b, _ in test_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        test_preds_list.extend(preds.cpu().numpy())

test_preds_scaled = np.array(test_preds_list).reshape(-1, 1)
test_preds_W = scaler_y.inverse_transform(test_preds_scaled).ravel()
test_targets_W = test_df[TARGET_COLUMN].values

# Predict on train set (for overfitting gap check)
train_val_dataset = TensorDataset(torch.FloatTensor(scaler_x.transform(train_val_df[FEATURE_COLUMNS].values)))
train_val_loader = DataLoader(train_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_preds_list = []
with torch.no_grad():
    for x_b, in train_val_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        train_preds_list.extend(preds.cpu().numpy())

train_preds_W = scaler_y.inverse_transform(np.array(train_preds_list).reshape(-1, 1)).ravel()
train_targets_W = train_val_df[TARGET_COLUMN].values

# Compute final benchmark metrics
test_mae = mean_absolute_error(test_targets_W, test_preds_W)
test_rmse = np.sqrt(mean_squared_error(test_targets_W, test_preds_W))
test_r2 = r2_score(test_targets_W, test_preds_W)

train_r2 = r2_score(train_targets_W, train_preds_W)
r2_gap = train_r2 - test_r2

print("=" * 55)
print(f"  PyTorch MLP Final Benchmark Results (Test Set)")
print("=" * 55)
print(f"  Test MAE:   {test_mae:,.1f} W")
print(f"  Test RMSE:  {test_rmse:,.1f} W")
print(f"  Test R²:    {test_r2:.4f}")
print(f"  Train R²:   {train_r2:.4f}")
print(f"  R² Gap:     {r2_gap:.4f}  ({'Minimal overfitting' if r2_gap < 0.1 else 'Overfitting observed'})")
print("=" * 55)

  PyTorch MLP Final Benchmark Results (Test Set)
  Test MAE:   3,469.3 W
  Test RMSE:  4,636.5 W
  Test R²:    0.6597
  Train R²:   0.8274
  R² Gap:     0.1677  (Overfitting observed)


## 9. Log Final Metrics, Prediction Plots & HParams to TensorBoard

In [9]:
# 1. Log metrics and HParams panel
tb.log_final(
    y_train=train_targets_W,
    y_pred_train=train_preds_W,
    y_test=test_targets_W,
    y_pred_test=test_preds_W,
    hparams={
        "architecture": "MLP (256-128-64)",
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_init": 1e-3,
        "dropout": 0.3,
        "optimizer": "AdamW",
        "output_act": "Linear"
    }
)

# 2. Log actual vs predicted time series and scatter plots
tb.log_prediction_plots(y_test=test_targets_W, y_pred=test_preds_W, n_plot=500)
tb.log_scatter_plot(y_test=test_targets_W, y_pred=test_preds_W)

# 3. Close writer
tb.close()
print("TensorBoard logging complete. Logs written to:", log_root)


[MLP] Train  MAE=1936W  R2=0.827
[MLP] Test   MAE=3469W  R2=0.660


[TB] Writer closed: 'MLP'
TensorBoard logging complete. Logs written to: ../runs


## 10. Committee Presentation Summary: Model Comparison

| Model | Model Family | Test MAE (W) | Test RMSE (W) | Test R² | Train-Test R² Gap |
|---|---|---|---|---|---|
| **LightGBM** | Gradient Boosted Trees | ~3,340 | ~4,580 | ~0.672 | 0.210 |
| **Extra Trees** | Randomized Ensembles | ~3,410 | ~4,690 | ~0.655 | 0.280 |
| **XGBoost** | Gradient Boosted Trees | 3,180 | 4,405 | 0.693 | 0.170 |
| **PyTorch MLP** | Deep Neural Network | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* |

Launch TensorBoard to inspect side-by-side:
```bash
tensorboard --logdir=runs/
```